In [2]:
# ============================================================
# Cell 1 - Imports
# ============================================================

import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

print("Libraries Loaded Successfully!")

Libraries Loaded Successfully!


In [3]:
# ============================================================
# Cell 2 - Load Dataset
# ============================================================

df = pd.read_csv("../results/reliability_dataset_v2.csv")

# Remove redundant features
X = df.drop(columns=[
    "prediction",
    "true_label",
    "failure_label",
    "msp_probability",
    "msp_ood_score"
])

y = df["failure_label"]

print("=" * 60)
print("UAIRE DATASET")
print("=" * 60)
print()

print("Samples :", X.shape[0])
print("Features:", X.shape[1])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print()
print("Training :", len(X_train))
print("Testing  :", len(X_test))

UAIRE DATASET

Samples : 10000
Features: 30

Training : 8000
Testing  : 2000


In [4]:
# ============================================================
# Cell 3 - Evaluation Function
# ============================================================

def evaluate_model(model, model_name):

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    y_prob = model.predict_proba(X_test)[:, 1]

    results = {

        "Model": model_name,

        "Accuracy": accuracy_score(y_test, y_pred),

        "Precision": precision_score(y_test, y_pred),

        "Recall": recall_score(y_test, y_pred),

        "F1": f1_score(y_test, y_pred),

        "ROC_AUC": roc_auc_score(y_test, y_prob)

    }

    return results

In [5]:
# ============================================================
# Cell 4 - Random Forest
# ============================================================

rf = RandomForestClassifier(

    n_estimators=300,

    random_state=42,

    class_weight="balanced",

    n_jobs=-1

)

rf_results = evaluate_model(

    rf,

    "Random Forest"

)

rf_results

{'Model': 'Random Forest',
 'Accuracy': 0.7525,
 'Precision': 0.5565092989985694,
 'Recall': 0.6777003484320557,
 'F1': 0.6111547525530243,
 'ROC_AUC': 0.8254651054825517}

In [6]:
# ============================================================
# Cell 5 - Logistic Regression
# ============================================================

lr = LogisticRegression(

    max_iter=1000,

    class_weight="balanced",

    random_state=42

)

lr_results = evaluate_model(

    lr,

    "Logistic Regression"

)

lr_results

/Users/omvdangi/Desktop/summer internship 2026 (VU)/MetaFailurePredictor/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


{'Model': 'Logistic Regression',
 'Accuracy': 0.7205,
 'Precision': 0.5084937712344281,
 'Recall': 0.7822299651567944,
 'F1': 0.6163349347975292,
 'ROC_AUC': 0.8112052914758761}

In [7]:
# ============================================================
# Cell 6 - XGBoost
# ============================================================

xgb = XGBClassifier(

    n_estimators=300,

    learning_rate=0.05,

    max_depth=6,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42,

    eval_metric="logloss"

)

xgb_results = evaluate_model(

    xgb,

    "XGBoost"

)

xgb_results

{'Model': 'XGBoost',
 'Accuracy': 0.772,
 'Precision': 0.6204081632653061,
 'Recall': 0.5296167247386759,
 'F1': 0.5714285714285714,
 'ROC_AUC': 0.8245121706877259}

In [8]:
# ============================================================
# Cell 7 - Model Comparison
# ============================================================

comparison = pd.DataFrame([

    rf_results,

    lr_results,

    xgb_results

])

comparison = comparison.sort_values(

    by="F1",

    ascending=False

)

comparison.reset_index(

    drop=True,

    inplace=True

)

comparison


,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.7205,0.508494,0.782230,0.616335,0.811205
1,Random Forest,0.7525,0.556509,0.677700,0.611155,0.825465
2,XGBoost,0.7720,0.620408,0.529617,0.571429,0.824512


In [9]:
# ============================================================
# Cell 8 - Save Results
# ============================================================

comparison.to_csv(

    "../results/model_comparison.csv",

    index=False

)

print("Results Saved Successfully!")

Results Saved Successfully!
